# うごイラ用: ComfyUI + Wan 2.2 を Colab で動かす

novelai-userscripts-example の「うごイラ → 動画モデルで動かす」から使う、リモートの GPU です。

1. メニューの「ランタイム → ランタイムのタイプを変更」で **L4**(か A100)を選ぶ
2. 上から順にセルを実行する(はじめはモデルの取得で 5〜10 分)
3. 最後のセルに出る 2 行を、アプリの `.env` に書いてバックエンドを起動し直す

終わったら「ランタイム → ランタイムを接続解除して削除」で止めてください(止めるまでコンピューティングユニットが減ります)。
詳しくは `docs/video_remote_jp.md`。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. ComfyUI とモデルを入れる

Wan 2.2 14B(fp8)の高ノイズ・低ノイズのモデルと、テキストエンコーダー・VAE で合わせて約 35GB です。

In [ ]:
%cd /content
!apt-get -qq install -y aria2 > /dev/null
![ -d ComfyUI ] || git clone -q --depth 1 https://github.com/comfyanonymous/ComfyUI
!pip -q install -r ComfyUI/requirements.txt

HF = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"
models = [
    ("diffusion_models", "wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors"),
    ("diffusion_models", "wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors"),
    ("text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
    ("vae", "wan_2.1_vae.safetensors"),
]
for folder, name in models:
    !aria2c -q -c -x 8 -d ComfyUI/models/{folder} {HF}/{folder}/{name}
!ls -la ComfyUI/models/diffusion_models ComfyUI/models/text_encoders ComfyUI/models/vae

## 2. ComfyUI と合言葉の中継を起動する

ComfyUI は Colab の中だけで待ち、外からは合言葉(トークン)を確かめる中継を通してだけ使えるようにします。

In [ ]:
import secrets, subprocess, time, urllib.request

BRANCH = "main"
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/hirotoitpost/novelai-userscripts-example/{BRANCH}/scripts/comfy_auth_proxy.py",
    "/content/comfy_auth_proxy.py",
)
TOKEN = globals().get("TOKEN") or secrets.token_urlsafe(32)
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI", stdout=open("/content/comfyui.log", "w"), stderr=subprocess.STDOUT,
)
proxy = subprocess.Popen(
    ["python", "/content/comfy_auth_proxy.py", "--token", TOKEN, "--listen", "127.0.0.1:8189"],
    stdout=open("/content/proxy.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        print("ComfyUI が起動しました")
        break
    except Exception:
        time.sleep(2)
else:
    print(open("/content/comfyui.log").read()[-3000:])

## 3. 外から入れるようにする(cloudflared)

URL はこのセルを実行するたびに変わります。

In [ ]:
import re, subprocess, threading, time, urllib.request

urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared"
)
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8189"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
start = time.time()
while url is None and time.time() - start < 60:
    line = tunnel.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
if url is None:
    tunnel.kill()
    raise RuntimeError(
        "60 秒たっても cloudflared の URL が取れませんでした。このセルをもう一度実行してください"
        "(続くときは Cloudflare 側の混雑か、Colab からの外への通信が止められています)。"
    )
# 出力を読み続けないと cloudflared が詰まって止まるので、残りは捨て続ける
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print("アプリの .env に書く 2 行:\n")
print(f"VIDEO_GEN_BASE_URL={url}")
print(f"VIDEO_GEN_TOKEN={TOKEN}")

## 4. 使い終わったら

このノートブックを開いたままにすると、使っていなくてもユニットが減ります。終わったら
「ランタイム → ランタイムを接続解除して削除」を選んでください。ログは `/content/comfyui.log` にあります。

In [ ]:
!tail -n 30 /content/comfyui.log